In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.3 Vocabulary design

> 🎯 **Goal:** Understand vocabulary size as a budget, and feel the trade-off between shorter sequences and a bigger model.

**Why this matters.** Choosing the vocab size is one of the first real design decisions in building an LLM, and it ripples through everything: model size, training cost, and how long your sequences are. There is no single right answer, only a trade you tune on purpose.

**The intuition.** Imagine packing a suitcase with labeled compartments. A few big compartments (small vocab) means you fold each shirt many times to fit, so packing takes more steps but the suitcase is light. Many small compartments (large vocab) means each shirt drops in whole, so packing is fast, but the suitcase itself is heavier and bulkier to carry. Vocab size is exactly this: more tokens encode text in fewer steps, but the model that has to carry every token grows.

**The mechanics.** Every token in the vocabulary costs two things. It needs one row in the *embedding table* (a learned vector, covered next lesson) and one slot in the output *logits* (the score the model produces per token when predicting). So a 600-token vocab is a literally bigger model than a 300-token vocab. In return, a bigger vocab has learned more merges, so it encodes the same text in fewer tokens, which means shorter sequences and cheaper attention. The job is picking the point where shorter sequences are worth the heavier model.

In [ ]:
from pocketlm import BPETokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
sample = corpus[:20000]
small = BPETokenizer().train(sample, 300)
big = BPETokenizer().train(sample, 600)
probe = corpus[20000:21000]
print("vocab 300 ->", len(small.encode(probe)), "tokens")
print("vocab 600 ->", len(big.encode(probe)), "tokens")

**What just happened.** The two prints encoded the same probe text with a 300-token vocab and a 600-token vocab. The 600-token vocab produced fewer (or equal) tokens. That is the budget paying off: a bigger vocabulary compresses the same text into a shorter sequence.

Why is it guaranteed to be fewer-or-equal, never more? Greedy BPE is *nested*. The 600-token vocab is built by continuing the exact same merge process the 300-token vocab used, then adding more merges. So it knows every shorthand the smaller vocab knew, plus extras. More shorthand can only shorten the encoding, which is what the assertion checks.

⚠️ **Common pitfalls**
- Chasing fewer tokens blindly: a giant vocab does shrink sequences, but it bloats the embedding table and the output layer, slows training, and starves rare tokens of examples to learn from. Bigger is not automatically better.
- Measuring on the training text: always probe compression on held-out text (here the probe slice is past the training slice). Measuring on the same text the tokenizer trained on overstates how well it compresses real input.

🏋️ **Try it yourself.** Add a third tokenizer trained to vocab `1200` and print its token count on the same probe. Then push the small one down to `260` (close to the 256 byte floor) and notice how little headroom it has to merge anything.

In [ ]:
assert len(big.encode(probe)) <= len(small.encode(probe))